# IMQCAM File Collections: Search, Inspect, and Download Data

This notebook shows modelers how to find IMQCAM data files using repository metadata and download only the matching results.

The main search workflow uses Girder's `/item/query` endpoint because it can filter directly on item metadata such as:

- **IGSN** (`meta.igsn`)
- **Data type** (`meta.data_type`)
- **Composition** (`meta.composition`)

You can use any one of these filters or combine them.

### IGSN note

Some IMQCAM items store more than one IGSN in the same metadata field, for example:

`JHABOX00001-002, JHABOX00001-001`

The helper functions below treat each value as a constituent IGSN so that searching for either specimen can return the item.

### Safe defaults

- Downloads are **off by default**.
- At least one metadata filter is required before `/item/query` is run.
- Downloading uses the same filtered `items` returned by the search.


## 1. Connect to the IMQCAM DMS

Run this notebook from the repository so that `imqcam.py` can be imported.


In [ ]:
import json
import re
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "imqcam.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))

from imqcam import get_client

client = get_client()
print(f"Connected to {client.urlBase}")


## 2. Explore available data types

`GET /imqcam/datatype` lists the data types currently available in IMQCAM.


In [ ]:
data_types = client.get("imqcam/datatype")
print(data_types)


## 3. Count files

`GET /imqcam/count` returns file counts. With no parameters, it returns overall counts.


In [ ]:
counts = client.get(
    "imqcam/count",
    parameters={},
)
print(counts)


### Count files for one IGSN

The count endpoint can also be restricted to a specimen IGSN.


In [ ]:
example_igsn = "CMXMAG00001-042"

counts_for_igsn = client.get(
    "imqcam/count",
    parameters={"igsn": example_igsn},
)
print(counts_for_igsn)


## 4. Optional: retrieve items for a known data type

`GET /imqcam/datafiles` is useful when the data type is already known.

**Important:** this endpoint requires `dataType`, so the more flexible metadata search in the next section uses `/item/query`.


In [ ]:
example_items = client.get(
    "imqcam/datafiles",
    parameters={"dataType": "ebsd_test"},
)
print(f"Returned {len(example_items)} item(s)")


## 5. Choose metadata filters

This is the main modeler-facing search workflow.

Set one or more filters below:

- `IGSN`: one IGSN, comma-separated IGSNs, or a Python list
- `DATA_TYPE`: for example `ebsd_test`
- `COMPOSITION_FILTER`: for example `IN718`

Leave a filter blank to ignore it.

### Example combinations

```python
IGSN = "JHABOX00001-001"
DATA_TYPE = ""
COMPOSITION_FILTER = ""
```

```python
IGSN = ""
DATA_TYPE = "ebsd_test"
COMPOSITION_FILTER = ""
```

```python
IGSN = ""
DATA_TYPE = ""
COMPOSITION_FILTER = "IN718"
```

```python
IGSN = ["JHABOX00001-001", "JHABOX00001-002"]
DATA_TYPE = "ebsd_test"
COMPOSITION_FILTER = "IN718"
```


In [ ]:
# Choose any combination of filters.
IGSN = ""
DATA_TYPE = "ebsd_test"
COMPOSITION_FILTER = ""

PAGE_SIZE = 100
MAX_RESULTS = None  # e.g. 500 for testing; None fetches all matching pages


## 6. Search helpers

These helpers:

1. normalize one or many requested IGSNs;
2. match constituent IGSNs safely when metadata contains comma-separated values;
3. combine active filters into one Mongo-style query;
4. paginate through `/item/query`.


In [ ]:
def split_igsns(value):
    """Normalize one or many IGSNs into a unique, ordered list."""
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        result = []
        for part in value:
            result.extend(split_igsns(part))
        return list(dict.fromkeys(result))

    text = str(value).strip()
    if not text or text.lower() in {"none", "nan"}:
        return []

    return list(
        dict.fromkeys(
            part.strip()
            for part in text.split(",")
            if part.strip()
        )
    )


def _exact_regex(value):
    """Case-insensitive exact match for an ordinary metadata field."""
    escaped = re.escape(str(value).strip())
    return {"$regex": rf"^{escaped}$", "$options": "i"}


def _igsn_regex(value):
    """
    Match one constituent IGSN when meta.igsn is stored as either:
      - one string;
      - a comma-separated string; or
      - an array/list of strings.

    The delimiter-aware pattern avoids accidental partial matches.
    """
    escaped = re.escape(str(value).strip())
    pattern = rf"(^|,\s*){escaped}(\s*,|$)"
    return {"$regex": pattern, "$options": "i"}


def build_item_query(igsn="", data_type="", composition=""):
    """Build a Mongo-style query from the active metadata filters."""
    clauses = []

    requested_igsns = split_igsns(igsn)
    data_type = str(data_type or "").strip()
    composition = str(composition or "").strip()

    if requested_igsns:
        igsn_clauses = [
            {"meta.igsn": _igsn_regex(value)}
            for value in requested_igsns
        ]

        clauses.append(
            igsn_clauses[0]
            if len(igsn_clauses) == 1
            else {"$or": igsn_clauses}
        )

    if data_type:
        clauses.append({"meta.data_type": _exact_regex(data_type)})

    if composition:
        clauses.append({"meta.composition": _exact_regex(composition)})

    if len(clauses) == 1:
        return clauses[0]

    return {"$and": clauses}


def query_items(
    client,
    igsn="",
    data_type="",
    composition="",
    page_size=100,
    max_results=None,
):
    """Query /item/query with one or more metadata filters."""
    if (
        not split_igsns(igsn)
        and not str(data_type or "").strip()
        and not str(composition or "").strip()
    ):
        raise ValueError(
            "Provide at least one filter: IGSN, DATA_TYPE, or COMPOSITION_FILTER."
        )

    query = build_item_query(
        igsn=igsn,
        data_type=data_type,
        composition=composition,
    )

    print("Mongo query:")
    print(json.dumps(query, indent=2))

    results = []
    offset = 0

    while True:
        limit = page_size

        if max_results is not None:
            remaining = max_results - len(results)
            if remaining <= 0:
                break
            limit = min(limit, remaining)

        batch = client.get(
            "item/query",
            parameters={
                "query": json.dumps(query),
                "limit": limit,
                "offset": offset,
            },
        )

        results.extend(batch)

        if len(batch) < limit:
            break

        offset += len(batch)

    return results


## 7. Run the search

The returned `items` are already restricted by the active metadata filters. The download section later uses these same filtered results.


In [ ]:
items = query_items(
    client=client,
    igsn=IGSN,
    data_type=DATA_TYPE,
    composition=COMPOSITION_FILTER,
    page_size=PAGE_SIZE,
    max_results=MAX_RESULTS,
)

print(f"\nMatched items: {len(items):,}")


## 8. Inspect matching metadata

Review the matches before downloading. The table below shows the key metadata fields for each returned item.


In [ ]:
rows = []

for item in items:
    meta = item.get("meta", {}) or {}

    rows.append(
        {
            "name": item.get("name"),
            "item_id": item.get("_id"),
            "igsn": meta.get("igsn"),
            "composition": meta.get("composition"),
            "data_type": meta.get("data_type"),
        }
    )

results_df = pd.DataFrame(
    rows,
    columns=["name", "item_id", "igsn", "composition", "data_type"],
)

display(results_df.head(20))


## 9. Download the matched items

Downloads are disabled by default.

Review `results_df` first. When the results are correct, set:

```python
DOWNLOAD_FILES = True
```

Only the `items` returned by the active metadata query are downloaded.

`DOWNLOAD_LIMIT` is set to `5` by default as an additional safeguard. Set it to `None` only when you intend to download every matched item.


In [ ]:
DOWNLOAD_FILES = False
DOWNLOAD_LIMIT = 5  # Set to None to download every matched item


def safe_folder_name(value, fallback):
    """Create a simple filesystem-safe folder component."""
    text = str(value or "").strip() or fallback
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text)


if DOWNLOAD_FILES:
    items_to_download = items

    if DOWNLOAD_LIMIT is not None:
        items_to_download = items_to_download[:DOWNLOAD_LIMIT]

    requested_igsns = split_igsns(IGSN)

    if not requested_igsns:
        igsn_folder = "all_igsns"
    elif len(requested_igsns) == 1:
        igsn_folder = requested_igsns[0]
    else:
        igsn_folder = "multiple_igsns"

    download_folder = (
        REPO_ROOT
        / "downloads"
        / safe_folder_name(DATA_TYPE, "all_data_types")
        / safe_folder_name(igsn_folder, "all_igsns")
        / safe_folder_name(COMPOSITION_FILTER, "all_compositions")
    )
    download_folder.mkdir(parents=True, exist_ok=True)

    print(f"Downloading {len(items_to_download):,} matched item(s)...")
    print(f"Destination: {download_folder}")

    for number, item in enumerate(items_to_download, start=1):
        item_id = item.get("_id")
        item_name = item.get("name", item_id)

        if not item_id:
            print(f"Skipping item without an _id: {item_name}")
            continue

        client.downloadItem(
            item_id,
            dest=str(download_folder),
        )

        print(f"[{number}/{len(items_to_download)}] {item_name}")

    print("Download complete.")

else:
    print(
        "Downloads are OFF. Review the matched items first, then set "
        "DOWNLOAD_FILES = True when ready."
    )


## Summary

For most modeler searches, use Sections 5–9:

1. choose at least one metadata filter;
2. run the `/item/query` search;
3. inspect the returned metadata;
4. enable downloads only after confirming the matches.

This keeps IGSN, data-type, and composition filtering in one workflow and ensures the download step uses the same filtered results shown in the notebook.
